# Design a Star Schema — Solution


## Grain Statement

The fact table grain is **one row per customer interaction** — each record represents a single voice call, chat session, or email exchange with a unique `interaction_id`.

## Star Schema

**Fact Table: `fact_interactions`**

| Column | Type | Description |
|--------|------|-------------|
| interaction_id | STRING (PK) | Unique interaction identifier |
| date_key | INT (FK) | → dim_date |
| customer_key | STRING (FK) | → dim_customer |
| agent_key | STRING (FK) | → dim_agent |
| channel | STRING | voice, chat, email |
| duration_sec | INT | Interaction length |
| resolution_status | STRING | resolved, escalated, pending |
| satisfaction_score | DECIMAL(3,1) | 0.0–5.0 |
| topic | STRING | billing, technical, onboarding |

**Dimension: `dim_date`**

| Column | Type |
|--------|------|
| date_key | INT (PK, YYYYMMDD) |
| full_date | DATE |
| day_of_week | STRING |
| week_number | INT |
| month | INT |
| quarter | INT |
| year | INT |
| is_weekend | BOOLEAN |

**Dimension: `dim_customer`**

| Column | Type |
|--------|------|
| customer_id | STRING (PK) |
| customer_name | STRING |
| industry | STRING |
| tier | STRING |

**Dimension: `dim_agent`**

| Column | Type |
|--------|------|
| agent_id | STRING (PK) |
| agent_name | STRING |
| team | STRING |

## Pre-Aggregated KPIs

| KPI | Formula | Why Pre-Aggregate |
|-----|---------|-------------------|
| `avg_handle_time_sec` | `AVG(duration_sec)` grouped by date + channel + team | Most-queried metric; avoids scanning millions of rows per dashboard refresh |
| `first_contact_resolution_pct` | `COUNT(status='resolved') / COUNT(*) * 100` grouped by date + channel | Complex filter logic that BI authors get wrong; standardize once in gold |
| `weekly_interaction_volume` | `COUNT(*)` grouped by week + channel + tier | Trend analysis requires consistent weekly bucketing; pre-computed avoids week-boundary errors |

## Refresh Cadence

**Every 4 hours.**

Rationale: Interactions are near-real-time (streaming), but the gold star schema serves BI dashboards that refresh 3–4x/day. A 4-hour cadence balances freshness (executives see same-day data) with compute cost (6 runs/day vs. 24 for hourly). The daily KPI rollup (`agent_performance`) refreshes once at 6 AM UTC for prior-day completeness.

## Mermaid ER Diagram

```mermaid
erDiagram
    dim_date ||--o{ fact_interactions : "date_key"
    dim_customer ||--o{ fact_interactions : "customer_key"
    dim_agent ||--o{ fact_interactions : "agent_key"

    fact_interactions {
        string interaction_id PK
        int date_key FK
        string customer_key FK
        string agent_key FK
        string channel
        int duration_sec
        string resolution_status
        decimal satisfaction_score
        string topic
    }

    dim_date {
        int date_key PK
        date full_date
        string day_of_week
        int week_number
        int month
        int quarter
        int year
        boolean is_weekend
    }

    dim_customer {
        string customer_id PK
        string customer_name
        string industry
        string tier
    }

    dim_agent {
        string agent_id PK
        string agent_name
        string team
    }
```